In [38]:
import anthropic
import os
from dotenv import main

main.load_dotenv()

client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_KEY"),
)

def return_response(prompt):
    message = client.messages.create(
        model="claude-3-7-sonnet-20250219",
        max_tokens=8192,
        messages=[
            {"role": "user", "content": prompt},
            {"role":"assistant", "content": "Here is the JSON requested:\n{"}
        ]
    )

    return message.content[0].text.strip()

In [39]:
return_response("What is the capital of France?")
# Output: "The capital of France is Paris."

'"capital": "Paris",\n  "country": "France"\n}'

In [40]:
import requests

def get_url_content_from_readme(identifier):
    url = f"https://raw.githubusercontent.com/modelcontextprotocol/servers/refs/heads/main/src/{identifier}/README.md"
    readme_content = requests.get(url).text
    return readme_content

In [41]:
def retrieve_tools_from_readme(identifier):
    readme_content = get_url_content_from_readme(identifier)
    get_me_tools_prompt = f'''
        Extract the tools and resources from the following README content.
        Both are clearly defined in the README.
        Provide them in a nice JSON format list with parameters -> name, description and arguments. arguments should also be a list of objects with name, type, description and required/not.
        Here is the README content:
        {readme_content}
    '''

    raw_response = return_response(get_me_tools_prompt)
    return raw_response

In [42]:
import json

def store_tools_in_file(identifier, tools):
    tools = "{" + tools
    with open(f"raw_data/{identifier}.json", "w") as file:
        json.dump(tools, file, indent=4)

In [43]:
def process(identifier):
    tools = retrieve_tools_from_readme(identifier)
    store_tools_in_file(identifier, tools)

In [44]:
idenitifer_list = ["filesystem", "postgres", "github", "sentry", "brave-search", "fetch", "puppeteer", "slack", "google-maps"]

In [45]:
process(idenitifer_list[0])

In [46]:
for identifier in idenitifer_list[1:]:
    process(identifier)